# SV2TTS — Speaker Verification to Text-to-Speech

**Paper:** [Transfer Learning from Speaker Verification to Multispeaker Text-to-Speech Synthesis](https://arxiv.org/abs/1806.04558) (Wan et al., Google, 2018)

Also known as: **Real-Time Voice Cloning (RTVC)**

---

## Core Idea

Train a **speaker encoder** separately on speaker verification, then freeze it and use its d-vectors to condition a multi-speaker TTS model.

This **decouples** two problems:
1. *Who is speaking?* — solved by the speaker encoder
2. *What are they saying?* — solved by the TTS model

Because the two components are trained independently, the speaker encoder can be trained on far more data (speaker verification datasets like VoxCeleb have thousands of speakers) than the TTS model.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
print("PyTorch:", torch.__version__)

## 1. Speaker Conditioning Methods

Once we have a d-vector, we need to inject it into the TTS acoustic model.
Three common strategies:

| Method | How | When to use |
|--------|-----|-------------|
| **Concatenate** | Append d-vector to each encoder output frame | Simple, widely used |
| **Add** | Element-wise add (after linear projection) to hidden states | Memory efficient |
| **AdaLN** | d-vector -> scale + shift for LayerNorm in each layer | Most expressive |

SV2TTS (original paper) uses **concatenation** at every decoder step of Tacotron 2.

In [ ]:
# Shared speaker encoder (from previous notebook — simplified here)
class SpeakerEncoder(nn.Module):
    def __init__(self, n_mels=40, d_hidden=128, n_layers=3, d_embed=128):
        super().__init__()
        self.lstm = nn.LSTM(n_mels, d_hidden, num_layers=n_layers, batch_first=True)
        self.proj = nn.Linear(d_hidden, d_embed)

    def forward(self, mel):
        _, (h_n, _) = self.lstm(mel)
        return F.normalize(self.proj(h_n[-1]), dim=-1)

# Method 1: Concatenate d-vector to encoder output
class ConcatConditionedEncoder(nn.Module):
    def __init__(self, vocab_size=100, d_model=128, d_embed=128):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc = nn.Embedding(2000, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model + d_embed, nhead=4, dim_feedforward=512,
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.proj = nn.Linear(d_model + d_embed, 80)

    def forward(self, phoneme_ids, spk_embed):
        B, T = phoneme_ids.shape
        pos = torch.arange(T, device=phoneme_ids.device).unsqueeze(0)
        x = self.embed(phoneme_ids) + self.pos_enc(pos)   # (B, T, d_model)
        # Concatenate speaker embedding to every time step
        spk = spk_embed.unsqueeze(1).expand(-1, T, -1)    # (B, T, d_embed)
        x = torch.cat([x, spk], dim=-1)                   # (B, T, d_model+d_embed)
        x = self.transformer(x)
        return self.proj(x)   # (B, T, 80) mel prediction

# Method 2: Add d-vector (after projection)
class AddConditionedEncoder(nn.Module):
    def __init__(self, vocab_size=100, d_model=128, d_embed=128):
        super().__init__()
        self.embed    = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc  = nn.Embedding(2000, d_model)
        self.spk_proj = nn.Linear(d_embed, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model, nhead=4, dim_feedforward=512,
            batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=2)
        self.proj = nn.Linear(d_model, 80)

    def forward(self, phoneme_ids, spk_embed):
        B, T = phoneme_ids.shape
        pos = torch.arange(T, device=phoneme_ids.device).unsqueeze(0)
        x = self.embed(phoneme_ids) + self.pos_enc(pos)
        # Add projected speaker embedding to all positions
        x = x + self.spk_proj(spk_embed).unsqueeze(1)
        x = self.transformer(x)
        return self.proj(x)

# Test both methods
spk_enc = SpeakerEncoder()
ref_mel = torch.randn(2, 160, 40)
spk_embed = spk_enc(ref_mel)

phonemes = torch.randint(1, 100, (2, 12))
enc1 = ConcatConditionedEncoder()
enc2 = AddConditionedEncoder()
out1 = enc1(phonemes, spk_embed)
out2 = enc2(phonemes, spk_embed)
print("Concat conditioning output:", out1.shape)
print("Add conditioning output:   ", out2.shape)

## 2. Multi-Speaker TTS Training

To train SV2TTS, we need a **multi-speaker dataset** where each utterance is paired with:
- The text / phoneme sequence
- The ground-truth mel spectrogram
- The speaker identity (used to extract d-vector from other utterances of the same speaker)

During training, the d-vector is extracted from a **different utterance** of the same speaker (not the one being synthesized). This forces the model to rely on the d-vector for speaker style rather than memorizing.

In [ ]:
class MultiSpeakerTTS(nn.Module):
    def __init__(self, vocab_size=100, d_model=128, d_embed=128, n_mels=80):
        super().__init__()
        self.speaker_encoder = SpeakerEncoder(n_mels=40, d_embed=d_embed)
        self.acoustic_model  = AddConditionedEncoder(vocab_size, d_model, d_embed)

    def forward(self, phoneme_ids, ref_mel):
        # ref_mel: reference audio from the target speaker (B, T_ref, 40)
        spk_embed = self.speaker_encoder(ref_mel)         # (B, d_embed)
        mel_pred  = self.acoustic_model(phoneme_ids, spk_embed)  # (B, T_text, 80)
        return mel_pred, spk_embed

model = MultiSpeakerTTS()

# Training step: speaker A says "hello" and "world"
# We synthesize "hello" conditioned on d-vector from "world" (different utterance, same speaker)
phonemes_hello = torch.randint(1, 100, (2, 10))
ref_mel_world  = torch.randn(2, 120, 40)   # reference utterance for d-vector
gt_mel_hello   = torch.randn(2, 10, 80)   # ground-truth mel for "hello"

mel_pred, spk_embed = model(phonemes_hello, ref_mel_world)
loss = F.l1_loss(mel_pred, gt_mel_hello)
print(f"Mel L1 loss: {loss.item():.4f}")
print(f"Speaker embed shape: {spk_embed.shape}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Zero-Shot Voice Cloning Pipeline

At inference, no retraining is needed. Just provide a short audio clip from the target speaker:

```
Reference Audio (3-30 sec from target speaker)
          |
    Feature Extraction
    (Log-Mel spectrogram, 40 bins)
          |
    Speaker Encoder (frozen)
          |
      d-vector (256-d)
          |
    TTS Acoustic Model
    (conditioned on d-vector)
          |
    Mel Spectrogram
          |
    Vocoder (HiFi-GAN)
          |
    Waveform in target speaker's voice
```

In [ ]:
# Full inference pipeline demo
def clone_voice(model, text_phonemes, ref_audio_mel, vocoder_proxy=None):
    model.eval()
    with torch.no_grad():
        mel_pred, spk_embed = model(text_phonemes, ref_audio_mel)
    return mel_pred, spk_embed

# Two different target speakers
spk_a_ref = torch.randn(1, 100, 40)   # Speaker A: 2.5 sec clip
spk_b_ref = torch.randn(1, 100, 40)   # Speaker B: 2.5 sec clip
text = torch.randint(1, 100, (1, 15)) # Same text for both

mel_a, embed_a = clone_voice(model, text, spk_a_ref)
mel_b, embed_b = clone_voice(model, text, spk_b_ref)

cos_sim = F.cosine_similarity(embed_a, embed_b).item()
print(f"Speaker A embedding: {embed_a.shape}")
print(f"Speaker B embedding: {embed_b.shape}")
print(f"Cosine similarity A vs B: {cos_sim:.4f}  (lower = more different)")

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
for ax, mel, title in zip(axes, [mel_a[0].T, mel_b[0].T], ["Speaker A", "Speaker B"]):
    ax.imshow(mel.numpy(), origin="lower", aspect="auto", cmap="magma")
    ax.set_title(f"Predicted Mel — {title}")
    ax.set_xlabel("Text frame")
    ax.set_ylabel("Mel bin")
plt.tight_layout()
plt.savefig("figures/sv2tts_mels.png", dpi=110, bbox_inches="tight")
plt.show()
print("Same text, different speaker embeddings -> different mel patterns")

## 4. Limitations of SV2TTS

| Limitation | Description |
|------------|-------------|
| **Two-stage** | Speaker encoder trained separately — not jointly optimized |
| **Mel bottleneck** | Vocoder needed after TTS; mel loses fine waveform details |
| **Quality vs. length** | Very short reference clips (<3 sec) give lower quality |
| **Language transfer** | Cloning across languages is unreliable |
| **Prosody gap** | d-vector captures timbre, but not speaking style / emotion well |

These limitations are addressed in later work (YourTTS, VALL-E).

## Summary

| Step | Component |
|------|-----------|
| 1. Pre-train speaker encoder | GE2E loss on VoxCeleb / LibriSpeech |
| 2. Freeze speaker encoder | Extract d-vectors for all training utterances |
| 3. Train multi-speaker TTS | Conditioned on d-vectors (Tacotron 2 / FastSpeech 2) |
| 4. Inference | Extract d-vector from 3-30 sec reference -> synthesize |

**Next:** VALL-E — instead of a speaker encoder, use audio tokens and a language model for in-context voice cloning.